In [0]:
import io
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
from PIL import Image
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader

import mlflow

ROOT = Path("/Volumes/workspace/default/glass_sample")

SAVE = ROOT / "models" / "rgbd_glass"
CACHE = Path("/tmp/rgbd_gsd_cache_224")

SIZE = 224
EPOCHS = 5
BATCH_SIZE = 8
LEARNING_RATE = 0.001
SEED = 42
PROGRESS_EVERY = 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

assert torch.cuda.is_available(), (
    "GPU NOT DETECTED. Select Serverless GPU"
    "and the AI environment in Databricks"
)

DEVICE = torch.device("cuda")
torch.backends.cudnn.benchmark = True

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)

SAVE.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

train_parquets = sorted(ROOT.glob("train-*.parquet"))
test_parquets = sorted(ROOT.glob("test-*.parquet"))

assert len(train_parquets) == 2, "Expected 2 files bro"
assert len(test_parquets) == 1, "Expected 1 file bro"

def format_time(seconds):
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def load_image(value):
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            return Image.open(
                io.BytesIO(value["bytes"])
            ).copy()

        path = value.get("path")
        if path and Path(path).is_file():
            return Image.open(path).copy()

    if isinstance(value, (bytes, bytearray, memoryview)):
        return Image.open(io.BytesIO(value)).copy()

    raise ValueError(f"unsupported image data: {type(value)}")


def prepare_sample(row, output_path):
    rgb_img = load_image(row["image"]).convert("RGB")
    depth_img = load_image(row["depth"])
    mask_img = load_image(row["mask"]).convert("L")

    assert rgb_img.size == depth_img.size == mask_img.size, (
        f"mismatched image dimensions: {row['image_id']}"
    )

    rgb = np.asarray(
        rgb_img.resize((SIZE, SIZE), Image.Resampling.BILINEAR),
        dtype=np.uint8
    )

    depth = np.asarray(
        depth_img.resize((SIZE, SIZE), Image.Resampling.NEAREST)
    )

    if depth.ndim == 3:
        if (
            depth.shape[-1] < 3
            or not np.array_equal(depth[..., 0], depth[..., 1])
            or not np.array_equal(depth[..., 0], depth[..., 2])
        ):
            raise ValueError(
                f"unexpected format: {row['image_id']}"
            )
        depth = depth[..., 0]

    depth = depth.astype(np.float32)

    valid = np.isfinite(depth) & (depth > 0)

    scale = (
        float(np.percentile(depth[valid], 99))
        if valid.any() else 1.0
    )
    scale = max(scale, 1e-6)

    normalized_depth = np.clip(
        np.where(valid, depth / scale, 0),
        0, 1
    ).astype(np.float16)

    raw_mask = np.asarray(
        mask_img.resize((SIZE, SIZE), Image.Resampling.NEAREST)
    )

    unique = np.unique(raw_mask)

    if not np.isin(unique, [0, 1, 255]).all():
        raise ValueError(
            f"Unexpected mask labels {unique[:15]} "
            f"for image {row['image_id']}"
        )

    mask = (raw_mask > 0).astype(np.uint8)

    np.savez_compressed(
        output_path,
        rgb=rgb,
        depth=normalized_depth,
        valid=valid.astype(np.uint8),
        mask=mask
    )

def prepare_split(parquet_files, split, expected_count):
    output_dir = CACHE / split
    output_dir.mkdir(parents=True, exist_ok=True)

    paths = []
    seen_ids = set()
    started = time.perf_counter()
    count = 0

    for parquet_path in parquet_files:
        parquet = pq.ParquetFile(str(parquet_path))

        for batch in parquet.iter_batches(
            batch_size=8,
            columns=["image_id", "image", "depth", "mask"]
        ):
            for row in batch.to_pylist():
                image_id = str(row["image_id"])

                if image_id in seen_ids:
                    raise ValueError(
                        f"Duplicate image ID: {image_id}"
                    )

                seen_ids.add(image_id)

                output_path = (
                    output_dir / f"{count:05d}.npz"
                )

                if not output_path.exists():
                    prepare_sample(row, output_path)

                paths.append(output_path)
                count += 1

                if count % 50 == 0 or count == expected_count:
                    elapsed = time.perf_counter() - started
                    eta = (
                        (expected_count - count)
                        * elapsed / max(count, 1)
                    )

                    print(
                        f"PREPROCESS {split}: "
                        f"{count}/{expected_count} | "
                        f"Elapsed {format_time(elapsed)} | "
                        f"ETA {format_time(eta)}",
                        flush=True
                    )

    assert count == expected_count, (
        f"Expected {expected_count} examples, found {count}"
    )

    return paths

print("\npreparing rgbd glass dataset")

all_train = prepare_split(
    train_parquets, "train", 2400
)

all_test = prepare_split(
    test_parquets, "test", 609
)

indices = np.random.default_rng(SEED).permutation(2400)

train_paths = [all_train[i] for i in indices[:2160]]
val_paths = [all_train[i] for i in indices[2160:]]
test_paths = all_test

print("\nDATA READY")
print("Training:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))

class RGBDGlassDataset(Dataset):
    def __init__(self, paths, augment=False):
        self.paths = paths
        self.augment = augment

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with np.load(self.paths[index]) as sample:
            rgb = sample["rgb"].astype(np.float32) / 255
            depth = sample["depth"].astype(np.float32)
            valid = sample["valid"].astype(np.float32)
            mask = sample["mask"].astype(np.float32)

        if self.augment and random.random() < 0.5:
            rgb, depth, valid, mask = [
                np.flip(a, axis=1).copy()
                for a in (rgb, depth, valid, mask)
            ]

        if self.augment:
            rgb = np.clip(
                rgb * random.uniform(0.9, 1.1),
                0, 1
            )
        x = np.concatenate([
            rgb,
            depth[..., None],
            valid[..., None]
        ], axis=-1).astype(np.float32)

        return (
            torch.from_numpy(x.transpose(2, 0, 1).copy()),
            torch.from_numpy(mask[None].copy())
        )


train_ds = RGBDGlassDataset(train_paths, augment=True)
val_ds = RGBDGlassDataset(val_paths)
test_ds = RGBDGlassDataset(test_paths)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

def conv_block(input_channels, output_channels):
    return nn.Sequential(
        nn.Conv2d(
            input_channels, output_channels,
            3, padding=1, bias=False
        ),
        nn.BatchNorm2d(output_channels),
        nn.ReLU(inplace=True),

        nn.Conv2d(
            output_channels, output_channels,
            3, padding=1, bias=False
        ),
        nn.BatchNorm2d(output_channels),
        nn.ReLU(inplace=True)
    )


class RGBDUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = conv_block(5, 16)
        self.enc2 = conv_block(16, 32)
        self.enc3 = conv_block(32, 64)

        self.middle = conv_block(64, 128)

        self.dec3 = conv_block(128 + 64, 64)
        self.dec2 = conv_block(64 + 32, 32)
        self.dec1 = conv_block(32 + 16, 16)

        self.output = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        a = self.enc1(x)
        b = self.enc2(F.avg_pool2d(a, 2))
        c = self.enc3(F.avg_pool2d(b, 2))

        d = self.middle(F.avg_pool2d(c, 2))

        d = F.interpolate(
            d, size=c.shape[-2:],
            mode="bilinear", align_corners=False
        )
        d = self.dec3(torch.cat([d, c], dim=1))

        d = F.interpolate(
            d, size=b.shape[-2:],
            mode="bilinear", align_corners=False
        )
        d = self.dec2(torch.cat([d, b], dim=1))

        d = F.interpolate(
            d, size=a.shape[-2:],
            mode="bilinear", align_corners=False
        )
        d = self.dec1(torch.cat([d, a], dim=1))

        return self.output(d)

def dice_loss(logits, targets):
    probs = torch.sigmoid(logits)

    intersection = (
        probs * targets
    ).sum(dim=(1, 2, 3))

    denominator = (
        probs.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    return (
        1 - (2 * intersection + 1) / (denominator + 1)
    ).mean()


def evaluate(model, loader):
    model.eval()

    tp = 0
    fp = 0
    fn = 0
    total_loss = 0.0
    batches = 0

    with torch.inference_mode():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(x)

            loss = (
                loss_fn(logits.float(), y)
                + dice_loss(logits.float(), y)
            )

            total_loss += loss.item()
            batches += 1

            prediction = torch.sigmoid(logits) >= 0.5
            truth = y >= 0.5

            tp += int((prediction & truth).sum().item())
            fp += int((prediction & ~truth).sum().item())
            fn += int((~prediction & truth).sum().item())

    union = tp + fp + fn

    return {
        "loss": total_loss / max(batches, 1),
        "iou": tp / max(union, 1),
        "dice": 2 * tp / max(2 * tp + fp + fn, 1),
        "recall": tp / max(tp + fn, 1),
        "precision": tp / max(tp + fp, 1)
    }

positive = 0
total = 0

for path in train_paths:
    with np.load(path) as sample:
        positive += int(sample["mask"].sum())
        total += int(sample["mask"].size)

assert positive > 0, "No glass pixels found in training data"

pos_weight = min(
    6.0,
    max(1.0, (total - positive) / positive)
)

print("\nGlass pixel fraction:", round(positive / total, 4))
print("Positive class weight:", round(pos_weight, 2))

model = RGBDUNet().to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(pos_weight, device=DEVICE)
)

grad_scaler = torch.amp.GradScaler("cuda")

best_iou = -1.0
training_started = time.perf_counter()
last_validation_seconds = 0.0

run = None

try:
    run = mlflow.start_run(
        run_name="rgbd_glass_gpu_5epochs",
        nested=mlflow.active_run() is not None
    )

    mlflow.log_params({
        "dataset": "RGBD-GSD",
        "architecture": "RGBDUNet_5channel",
        "epochs": EPOCHS,
        "image_size": SIZE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "training_samples": len(train_ds),
        "validation_samples": len(val_ds),
        "test_samples": len(test_ds),
        "GPU": torch.cuda.get_device_name(0)
    })

except Exception as error:
    print("MLflow warning", error)

try:
    for epoch in range(1, EPOCHS + 1):
        model.train()

        epoch_started = time.perf_counter()
        cumulative_loss = 0.0

        for step, (x, y) in enumerate(train_loader, 1):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(x)

                loss = (
                    loss_fn(logits.float(), y)
                    + dice_loss(logits.float(), y)
                )

            grad_scaler.scale(loss).backward()
            grad_scaler.step(optimizer)
            grad_scaler.update()

            cumulative_loss += loss.item()

            # ETA UPDATE EVERY 10 BATCHES
            if (
                step % PROGRESS_EVERY == 0
                or step == len(train_loader)
            ):
                torch.cuda.synchronize()

                elapsed_epoch = (
                    time.perf_counter() - epoch_started
                )

                avg_batch_time = elapsed_epoch / step

                remaining_batches = (
                    len(train_loader) - step
                    + (EPOCHS - epoch) * len(train_loader)
                )

                estimated_remaining = (
                    remaining_batches * avg_batch_time
                    + (EPOCHS - epoch + 1)
                    * last_validation_seconds
                )

                total_elapsed = (
                    time.perf_counter() - training_started
                )

                print(
                    f"Epoch {epoch}/{EPOCHS} | "
                    f"Batch {step}/{len(train_loader)} | "
                    f"Loss {cumulative_loss / step:.4f} | "
                    f"Elapsed {format_time(total_elapsed)} | "
                    f"ETA {format_time(estimated_remaining)}",
                    flush=True
                )

        print(
            f"\nvalidating epoch {epoch}...",
            flush=True
        )

        validation_started = time.perf_counter()

        metrics = evaluate(model, val_loader)

        torch.cuda.synchronize()

        last_validation_seconds = (
            time.perf_counter() - validation_started
        )

        avg_train_loss = (
            cumulative_loss / len(train_loader)
        )

        print(
            f"\nEPOCH {epoch}/{EPOCHS} COMPLETE\n"
            f"Train loss: {avg_train_loss:.4f}\n"
            f"Validation IoU: {metrics['iou']:.4f}\n"
            f"Validation Dice: {metrics['dice']:.4f}\n"
            f"Validation Recall: {metrics['recall']:.4f}\n"
            f"Validation Precision: {metrics['precision']:.4f}\n",
            flush=True
        )

        if run is not None:
            mlflow.log_metrics({
                "train_loss": avg_train_loss,
                "val_iou": metrics["iou"],
                "val_dice": metrics["dice"],
                "val_recall": metrics["recall"],
                "val_precision": metrics["precision"]
            }, step=epoch)

        if metrics["iou"] > best_iou:
            best_iou = metrics["iou"]

            torch.save({
                "state_dict": model.state_dict(),
                "epoch": epoch,
                "val_iou": best_iou,
                "size": SIZE,
                "input_channels": [
                    "R", "G", "B",
                    "normalized_depth",
                    "depth_validity"
                ]
            }, SAVE / "glass_rgbd_best.pt")

            print(
                "Saved better checkpoint",
                flush=True
            )

    print(
        "\nALL 5 EPOCHS COMPLETED",
        "Total training time:",
        format_time(time.perf_counter() - training_started)
    )

    checkpoint = torch.load(
        SAVE / "glass_rgbd_best.pt",
        map_location="cpu",
        weights_only=True
    )

    model.load_state_dict(checkpoint["state_dict"])
    model.eval()

    print("\nEvaluating.............")

    test_metrics = evaluate(model, test_loader)

    print("FINAL TEST RESULTS:")
    for name, value in test_metrics.items():
        print(name, round(value, 4))

    if run is not None:
        mlflow.log_metrics({
            f"test_{name}": value
            for name, value in test_metrics.items()
        })

    cpu_model = RGBDUNet().cpu().eval()
    cpu_model.load_state_dict(checkpoint["state_dict"])

    with torch.inference_mode():
        traced_model = torch.jit.trace(
            cpu_model,
            torch.zeros(1, 5, SIZE, SIZE)
        )

    traced_model.save(
        str(SAVE / "glass_rgbd_traced.pt")
    )

    config = {
        "model": "RGBDUNet",
        "input_shape_nchw": [1, 5, SIZE, SIZE],
        "channels": [
            "red", "green", "blue",
            "normalized_depth", "depth_validity"
        ],
        "rgb_range": "[0,1]",
        "depth_normalization": (
            "Divide positive depth readings by each "
            "frame's 99th percentile. Clip to [0,1]."
        ),
        "validity": (
            "1 for finite positive depth, 0 otherwise"
        ),
        "output": (
            "Single channel of logits. Apply sigmoid; "
            "threshold 0.5 for binary glass mask."
        ),
        "dataset": "RGBD-GSD",
        "dataset_license": "CC BY-NC 4.0",
        "note": (
            "Requires validation on iPhone LiDAR data "
            "and conversion to Core ML"
        )
    }

    (SAVE / "model_config.json").write_text(
        json.dumps(config, indent=2)
    )

    sample_x, actual_mask = test_ds[0]

    with torch.inference_mode():
        probability = torch.sigmoid(
            model(
                sample_x.unsqueeze(0).to(DEVICE)
            )
        )[0, 0].cpu().numpy()

    predicted_mask = probability >= 0.5

    rgb = sample_x[:3].permute(1, 2, 0).numpy()

    overlay = rgb.copy()
    overlay[predicted_mask] = (
        0.45 * overlay[predicted_mask]
        + 0.55 * np.array([0.65, 0.25, 1.0])
    )

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    axes[0].imshow(rgb)
    axes[0].set_title("RGB image")

    axes[1].imshow(
        actual_mask[0].numpy(),
        cmap="gray",
        vmin=0,
        vmax=1
    )
    axes[1].set_title("Actual glass mask")

    axes[2].imshow(overlay)
    axes[2].set_title("Predicted glass overlay")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()

    fig.savefig(
        SAVE / "heldout_preview.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

    if run is not None:
        for filename in [
            "glass_rgbd_best.pt",
            "glass_rgbd_traced.pt",
            "model_config.json",
            "heldout_preview.png"
        ]:
            mlflow.log_artifact(str(SAVE / filename))

    print("\nGLASS MODEL TRAINING FINISHED!!!!!!!!!!!")
    print("Saved model files to:", SAVE)

finally:
    if run is not None:
        mlflow.end_run()

In [0]:
from pathlib import Path

ROOT = Path("/Volumes/workspace/default/glass_sample")

print("SEARCHING:", ROOT)
print("=" * 60)

for file in ROOT.rglob("*"):
    if file.is_file():
        size_mb = file.stat().st_size / (1024 * 1024)
        print(f"{file}  ({size_mb:.1f} MB)")

print("\n" + "=" * 60)
print("PARQUET FILES FOUND:")

parquet_files = sorted(ROOT.rglob("*.parquet"))

for file in parquet_files:
    print(file)

print(f"\nTotal Parquet files: {len(parquet_files)}")

In [0]:
import os
import importlib.util

print("Current folder:", os.getcwd())
print("MLflow location:", importlib.util.find_spec("mlflow"))

In [0]:
import mlflow
import torch

print("MLflow version:", mlflow.__version__)
print("MLflow location:", mlflow.__file__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [0]:
import traceback
import importlib.metadata

for package in ["mlflow", "mlflow-skinny", "mlflow-tracing"]:
    try:
        print(package, importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        print(package, "not installed")

try:
    import mlflow
    print("MLflow imported successfully")
except Exception:
    traceback.print_exc()

In [0]:
%pip install --upgrade "mlflow[databricks]==3.17.2"

In [0]:
%restart_python

In [0]:
import mlflow
import torch

print("MLflow:", mlflow.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [0]:
%pip install torch torchvision

In [0]:
dbutils.library.restartPython()

In [0]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

In [0]:
%pip install "transformers==4.57.6" "safetensors"

In [0]:
%restart_python

In [0]:
import io, json, os, random, time
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import SegformerForSemanticSegmentation
import mlflow

ROOT = Path('/Volumes/workspace/default/glass_sample')
SAVE = ROOT / 'models' / 'rgbd_glass_segformer_fusion'
CACHE = Path('/tmp/rgbd_gsd_cache_224')  
PRETRAINED = 'nvidia/segformer-b0-finetuned-ade-512-512'
SIZE = 224
EPOCHS = 15
BATCH_SIZE = 8
ENCODER_LR = 3e-5
HEAD_LR = 3e-4
SEED = 42
PROGRESS_EVERY = 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select a Databricks Serverless GPU environment first'
DEVICE = torch.device('cuda')
SAVE.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(min(4, os.cpu_count() or 4))
torch.backends.cudnn.benchmark = True
print('GPU:', torch.cuda.get_device_name(0), 'PyTorch:', torch.__version__, flush=True)
print('Model: pretrained SegFormer-B0 + depth-validity-gated fusion', flush=True)
print('Training epochs:', EPOCHS, 'Batch:', BATCH_SIZE, 'Image:', SIZE, flush=True)

def clock(s):
    s = max(0, int(s))
    return f'{s//3600:02d}:{s%3600//60:02d}:{s%60:02d}'

train_files = sorted(ROOT.glob('train-*.parquet'))
test_files = sorted(ROOT.glob('test-*.parquet'))
assert len(train_files) == 2 and len(test_files) == 1, (
    f'Expected 2 training + 1 test Parquet shards directly inside {ROOT}. '
    f'Found {train_files} and {test_files}'
)

def open_picture(value):
    if isinstance(value, dict):
        if value.get('bytes') is not None:
            return Image.open(io.BytesIO(value['bytes'])).copy()
        if value.get('path') and Path(value['path']).is_file():
            return Image.open(value['path']).copy()
    if isinstance(value, (bytes, bytearray, memoryview)):
        return Image.open(io.BytesIO(value)).copy()
    raise ValueError(f'Unrecognized Parquet image object: {type(value)}')

def cache_row(row, target):
    rgb_image = open_picture(row['image']).convert('RGB')
    depth_image = open_picture(row['depth'])
    mask_image = open_picture(row['mask']).convert('L')
    assert rgb_image.size == depth_image.size == mask_image.size, (
        f'RGB/depth/mask mismatch: image_id={row["image_id"]}'
    )
    rgb = np.asarray(rgb_image.resize((SIZE, SIZE), Image.Resampling.BILINEAR), dtype=np.uint8)
    depth = np.asarray(depth_image.resize((SIZE, SIZE), Image.Resampling.NEAREST))
    if depth.ndim == 3:
        if depth.shape[2] < 3 or not (
            np.array_equal(depth[...,0], depth[...,1]) and
            np.array_equal(depth[...,0], depth[...,2])
        ):
            raise ValueError(f'Unexpected color-coded depth: {row["image_id"]}')
        depth = depth[...,0]
    depth = depth.astype(np.float32)
    valid = np.isfinite(depth) & (depth > 0)
    scale = float(np.percentile(depth[valid], 99)) if valid.any() else 1.0
    scale = max(scale, 1e-6)
    norm_depth = np.clip(np.where(valid, depth / scale, 0), 0, 1).astype(np.float16)
    original_mask = np.asarray(mask_image.resize((SIZE, SIZE), Image.Resampling.NEAREST))
    unique = np.unique(original_mask)
    if not np.isin(unique, [0, 1, 255]).all():
        raise ValueError(f'Unexpected mask values for {row["image_id"]}: {unique[:15]}')
    mask = (original_mask > 0).astype(np.uint8)
    np.savez_compressed(target, rgb=rgb, depth=norm_depth,
                        valid=valid.astype(np.uint8), mask=mask)

def prepare(parquets, folder_name, n_expected):
    outdir = CACHE / folder_name
    outdir.mkdir(parents=True, exist_ok=True)
    files, seen = [], set()
    start = time.perf_counter()
    n = 0
    for parquet_path in parquets:
        parquet = pq.ParquetFile(str(parquet_path))
        for batch in parquet.iter_batches(batch_size=8,
                                          columns=['image_id','image','depth','mask']):
            for row in batch.to_pylist():
                image_id = str(row['image_id'])
                if image_id in seen:
                    raise ValueError(f'Duplicate {folder_name} image_id: {image_id}')
                seen.add(image_id)
                target = outdir / f'{n:05d}.npz'
                if not target.exists():
                    cache_row(row, target)
                files.append(target)
                n += 1
                if n % 100 == 0 or n == n_expected:
                    elapsed = time.perf_counter() - start
                    print(f'PREP {folder_name}: {n}/{n_expected} | '
                          f'ETA {clock(elapsed / n * (n_expected - n))}', flush=True)
    assert n == n_expected, f'Expected {n_expected} rows, found {n}'
    return files

print('\nPreparing RGB-D pairs (reuses prior cache if present)...', flush=True)
all_train = prepare(train_files, 'train', 2400)
all_test = prepare(test_files, 'test', 609)
order = np.random.default_rng(SEED).permutation(2400)
train_paths = [all_train[i] for i in order[:2160]]
val_paths = [all_train[i] for i in order[2160:]]
print('DATA READY | train:', len(train_paths), 'val:', len(val_paths),
      'test:', len(all_test), flush=True)

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
class GlassDataset(Dataset):
    def __init__(self, paths, augment=False):
        self.paths, self.augment = paths, augment
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        with np.load(self.paths[i]) as z:
            rgb = z['rgb'].astype(np.float32) / 255.0
            depth = z['depth'].astype(np.float32)
            valid = z['valid'].astype(np.float32)
            mask = z['mask'].astype(np.float32)
        if self.augment:
            if random.random() < 0.5:
                rgb, depth, valid, mask = [np.flip(t, axis=1).copy()
                                           for t in (rgb, depth, valid, mask)]
            rgb = np.clip(rgb * random.uniform(0.85, 1.15), 0, 1)
        normalized_rgb = (rgb - MEAN) / STD
        x = np.concatenate([normalized_rgb, depth[..., None], valid[..., None]], axis=-1)
        return (torch.from_numpy(x.transpose(2,0,1).copy()),
                torch.from_numpy(mask[None].copy()))

train_ds = GlassDataset(train_paths, augment=True)
val_ds = GlassDataset(val_paths)
test_ds = GlassDataset(all_test)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        pin_memory=True, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         pin_memory=True, num_workers=0)

positive = total = 0
for filename in train_paths:
    with np.load(filename) as z:
        positive += int(z['mask'].sum())
        total += int(z['mask'].size)
assert positive > 0
positive_weight = float(np.clip(np.sqrt((total-positive)/positive), 1.0, 3.0))
print('Glass fraction:', round(positive/total, 4),
      'BCE glass weight:', round(positive_weight, 3), flush=True)

class DepthAwareSegFormer(nn.Module):
    def __init__(self, pretrained_name=PRETRAINED):
        super().__init__()
        base = SegformerForSemanticSegmentation.from_pretrained(
            pretrained_name, use_safetensors=True
        )
        self.rgb_encoder = base.segformer
        widths = list(base.config.hidden_sizes) 
        projection_width = 64
        fused_width = 96
        self.rgb_projections = nn.ModuleList([
            nn.Sequential(nn.Conv2d(w, projection_width, 1),
                          nn.GroupNorm(8, projection_width), nn.GELU())
            for w in widths
        ])
        self.rgb_fuse = nn.Sequential(
            nn.Conv2d(4*projection_width, fused_width, 3, padding=1),
            nn.GroupNorm(8, fused_width), nn.GELU()
        )
        self.depth_encoder = nn.Sequential(
            nn.Conv2d(2, 32, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(64, fused_width, 3, padding=1),
            nn.GroupNorm(8, fused_width), nn.GELU()
        )
        self.depth_gate = nn.Sequential(
            nn.Conv2d(fused_width*2+1, fused_width, 3, padding=1),
            nn.Sigmoid()
        )
        self.classifier = nn.Sequential(
            nn.Conv2d(fused_width*2, fused_width, 3, padding=1),
            nn.GroupNorm(8, fused_width), nn.GELU(),
            nn.Dropout2d(0.1), nn.Conv2d(fused_width, 1, 1)
        )

    def forward(self, x):
        rgb = x[:, :3]      
        depth = x[:, 3:4]   
        valid = x[:, 4:5]   
        hidden = self.rgb_encoder(pixel_values=rgb,
                                  output_hidden_states=True,
                                  return_dict=True).hidden_states
        target = hidden[0].shape[-2:]
        multiscale = [F.interpolate(proj(h), size=target, mode='bilinear',
                                    align_corners=False)
                      for proj,h in zip(self.rgb_projections, hidden)]
        rgb_features = self.rgb_fuse(torch.cat(multiscale, dim=1))
        depth_features = self.depth_encoder(torch.cat([depth,valid], dim=1))
        depth_features = F.interpolate(depth_features, size=target,
                                       mode='bilinear', align_corners=False)
        valid_small = F.interpolate(valid, size=target, mode='nearest')
        gate = self.depth_gate(torch.cat([rgb_features,depth_features,valid_small],dim=1))
        fused = torch.cat([rgb_features, gate*depth_features], dim=1)
        logits_small = self.classifier(fused)
        return F.interpolate(logits_small, size=x.shape[-2:],
                             mode='bilinear', align_corners=False)

print('\nDownloading/loading pretrained SegFormer-B0...', flush=True)
model = DepthAwareSegFormer().to(DEVICE)
with torch.inference_mode():
    smoke = model(torch.zeros(1,5,SIZE,SIZE,device=DEVICE))
assert tuple(smoke.shape) == (1,1,SIZE,SIZE), tuple(smoke.shape)
print('MODEL CHECK PASSED', tuple(smoke.shape), flush=True)

bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([positive_weight],device=DEVICE))
def dice_loss(logits, truth):
    probability = torch.sigmoid(logits)
    intersection = (probability*truth).sum(dim=(1,2,3))
    denominator = probability.sum(dim=(1,2,3)) + truth.sum(dim=(1,2,3))
    return (1-(2*intersection+1)/(denominator+1)).mean()

def total_loss(logits, truth):
    return bce(logits.float(),truth) + dice_loss(logits.float(),truth)

@torch.inference_mode()
def evaluate(loader, threshold=0.5):
    model.eval()
    tp=fp=fn=0
    losses=[]
    for x,y in loader:
        x=x.to(DEVICE,non_blocking=True)
        y=y.to(DEVICE,non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits=model(x)
        losses.append(float(total_loss(logits,y).item()))
        pred=torch.sigmoid(logits.float()) >= threshold
        gt=y >= 0.5
        tp += int((pred & gt).sum().item())
        fp += int((pred & ~gt).sum().item())
        fn += int((~pred & gt).sum().item())
    return {'loss':float(np.mean(losses)),
            'iou':tp/max(tp+fp+fn,1),
            'dice':2*tp/max(2*tp+fp+fn,1),
            'recall':tp/max(tp+fn,1),
            'precision':tp/max(tp+fp,1)}

encoder_parameters = list(model.rgb_encoder.parameters())
head_parameters = [p for name,p in model.named_parameters()
                   if not name.startswith('rgb_encoder.')]
optimizer = torch.optim.AdamW([
    {'params':encoder_parameters,'lr':ENCODER_LR},
    {'params':head_parameters,'lr':HEAD_LR}
],weight_decay=1e-4)
scaler=torch.amp.GradScaler('cuda')

run=None
try:
    run=mlflow.start_run(run_name='rgbd_glass_segformer_b0_depth_fusion',
                         nested=mlflow.active_run() is not None)
    mlflow.log_params({'architecture':'SegFormer-B0_RGB_pretrained_depth_gated_fusion',
                       'pretrained_checkpoint':PRETRAINED,'size':SIZE,
                       'epochs':EPOCHS,'batch_size':BATCH_SIZE,'encoder_lr':ENCODER_LR,
                       'head_lr':HEAD_LR,'training_examples':2160,
                       'validation_examples':240,'test_examples':609,
                       'positive_weight':positive_weight,
                       'device':torch.cuda.get_device_name(0)})
    print('MLflow run:',run.info.run_id,flush=True)
except Exception as exc:
    print('MLflow warning; training continues:',str(exc),flush=True)

best_iou=-1
best_epoch=0
start=time.perf_counter()
last_val_sec=0
try:
    for epoch in range(1,EPOCHS+1):
        model.train()
        epoch_begin=time.perf_counter()
        rolling_loss=0.0
        for batch_number,(x,y) in enumerate(train_loader,1):
            x=x.to(DEVICE,non_blocking=True)
            y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16):
                logits=model(x)
                loss=total_loss(logits,y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer)
            scaler.update()
            rolling_loss += float(loss.item())
            if batch_number%PROGRESS_EVERY==0 or batch_number==len(train_loader):
                torch.cuda.synchronize()
                batch_avg=(time.perf_counter()-epoch_begin)/batch_number
                remaining=(len(train_loader)-batch_number)+(EPOCHS-epoch)*len(train_loader)
                eta=batch_avg*remaining + (EPOCHS-epoch+1)*last_val_sec
                print(f'EPOCH {epoch}/{EPOCHS} | batch {batch_number}/{len(train_loader)}'
                      f' | train_loss {rolling_loss/batch_number:.4f}'
                      f' | elapsed {clock(time.perf_counter()-start)}'
                      f' | ETA {clock(eta)}',flush=True)
        print('Validating epoch',epoch,'...',flush=True)
        vstart=time.perf_counter()
        val=evaluate(val_loader)
        last_val_sec=time.perf_counter()-vstart
        print(f'EPOCH {epoch} | val IoU {val["iou"]:.4f}'
              f' | Dice {val["dice"]:.4f}'
              f' | Precision {val["precision"]:.4f}'
              f' | Recall {val["recall"]:.4f}',flush=True)
        if run:
            mlflow.log_metrics({'train_loss':rolling_loss/len(train_loader),
                                **{f'val_{k}':v for k,v in val.items()}}, step=epoch)
        if val['iou']>best_iou:
            best_iou=val['iou']
            best_epoch=epoch
            torch.save({'state_dict':model.state_dict(),'epoch':epoch,'best_val_iou':best_iou,
                        'architecture':'DepthAwareSegFormer','pretrained':PRETRAINED,
                        'size':SIZE},SAVE/'glass_segformer_fusion_best.pt')
            print('SAVED NEW BEST CHECKPOINT',flush=True)

    checkpoint=torch.load(SAVE/'glass_segformer_fusion_best.pt',
                          map_location='cpu',weights_only=True)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    print('\nSelecting decision threshold on validation data..........',flush=True)
    candidates=[0.35,0.40,0.45,0.50,0.55,0.60,0.65,0.70]
    threshold_scores={t:evaluate(val_loader,t)['iou'] for t in candidates}
    chosen_threshold=max(candidates,key=lambda t:threshold_scores[t])
    print('Validation thresholds:',threshold_scores,flush=True)
    print('Chosen threshold:',chosen_threshold,flush=True)

    print('\nEvaluating untouched 609-image test set.....',flush=True)
    test=evaluate(test_loader,chosen_threshold)
    print('HELD-OUT TEST METRICS:',{k:round(v,4) for k,v in test.items()},flush=True)
    if run:
        mlflow.log_metrics({'best_validation_iou_at_0_5':best_iou,
                            'chosen_threshold':chosen_threshold,
                            **{f'test_{k}':v for k,v in test.items()}})

    config={'architecture':'pretrained_SegFormer_B0_plus_depth_validity_gated_fusion',
            'pretrained':PRETRAINED,'input_shape_nchw':[1,5,SIZE,SIZE],
            'input_channels':['R_imagenet_normalized','G_imagenet_normalized',
                              'B_imagenet_normalized','depth_p99_normalized','depth_validity'],
            'rgb_raw_range':'RGB bytes / 255 before ImageNet normalization',
            'rgb_mean':MEAN.tolist(),'rgb_std':STD.tolist(),
            'depth_normalization':'valid if finite and >0; per-frame p99 of positive values; clip depth/p99 to [0,1]; invalid depth=0',
            'depth_validity':'1=positive measured depth; 0=missing',
            'output':'1x1x224x224 logits; sigmoid for glass probability',
            'threshold':chosen_threshold,'best_epoch':best_epoch,
            'best_validation_iou_at_0_5':best_iou,'heldout_test_metrics':test,
            'training_examples':2160,'validation_examples':240,'test_examples':609,
            'seed':SEED,'source':'garrying/RGBD-GSD',
            'warning':'Experimental dataset model, not validated on iPhone LiDAR or as a safety device. Missing depth is not proof of glass.'}
    (SAVE/'model_config.json').write_text(json.dumps(config,indent=2))

    fig,axes=plt.subplots(3,4,figsize=(13,10))
    for row,index in enumerate([0,1,2]):
        x,ground_truth=test_ds[index]
        with torch.inference_mode(),torch.autocast(device_type='cuda',dtype=torch.float16):
            prob=torch.sigmoid(model(x[None].to(DEVICE)).float())[0,0].cpu().numpy()
        rgb=(x[:3].permute(1,2,0).numpy()*STD+MEAN).clip(0,1)
        mask=prob>=chosen_threshold
        overlay=rgb.copy()
        overlay[mask]=0.4*overlay[mask]+0.6*np.array([0.8,0.15,0.95])
        axes[row,0].imshow(rgb)
        axes[row,1].imshow(x[3].numpy(),cmap='gray',vmin=0,vmax=1)
        axes[row,2].imshow(ground_truth[0].numpy(),cmap='gray',vmin=0,vmax=1)
        axes[row,3].imshow(overlay)
        for ax in axes[row]: ax.axis('off')
    for ax,title in zip(axes[0],['RGB image','Normalized depth','Ground-truth glass','Predicted glass (purple)']):
        ax.set_title(title)
    plt.tight_layout()
    preview=SAVE/'glass_comparison.png'
    fig.savefig(preview,dpi=150,bbox_inches='tight')
    plt.show()

    if run:
        for path in [SAVE/'glass_segformer_fusion_best.pt',SAVE/'model_config.json',preview]:
            mlflow.log_artifact(str(path))
    print('\nCOMPLETE. Model files:',SAVE,flush=True)
    print('Best epoch:',best_epoch,'Best validation IoU (0.5):',round(best_iou,4),flush=True)
    print('Held-out test IoU:',round(test['iou'],4),flush=True)
finally:
    if run:
        mlflow.end_run()


In [0]:

import time
from pathlib import Path
import torch
import mlflow

BEST_PATH = SAVE / "glass_segformer_fusion_best.pt"
FINETUNED_PATH = SAVE / "glass_segformer_fusion_finetuned.pt"

EXTRA_EPOCHS = 10
PATIENCE = 4
MIN_IMPROVEMENT = 0.001

checkpoint = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=True
)
model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE)

baseline = evaluate(val_loader, threshold=0.5)
best_val_iou = baseline["iou"]
bad_epochs = 0
best_path = BEST_PATH

print(f"Starting from validation IoU: {best_val_iou:.4f}")
print(f"Original best epoch: {checkpoint['epoch']}")

fine_optimizer = torch.optim.AdamW(
    [
        {
            "params": model.rgb_encoder.parameters(),
            "lr": 1e-5
        },
        {
            "params": [
                p for name, p in model.named_parameters()
                if not name.startswith("rgb_encoder.")
            ],
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

fine_scaler = torch.amp.GradScaler("cuda")
start = time.perf_counter()

with mlflow.start_run(run_name="glass_segformer_finetuning"):
    mlflow.log_params({
        "starting_checkpoint": BEST_PATH.name,
        "additional_epochs_max": EXTRA_EPOCHS,
        "early_stopping_patience": PATIENCE,
        "encoder_learning_rate": 1e-5,
        "fusion_learning_rate": 1e-4,
        "baseline_validation_iou": best_val_iou,
    })

    for epoch in range(1, EXTRA_EPOCHS + 1):
        model.train()
        total_train_loss = 0.0

        for batch_num, (x, y) in enumerate(train_loader, 1):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            fine_optimizer.zero_grad(set_to_none=True)

            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(x)
                loss = total_loss(logits, y)

            fine_scaler.scale(loss).backward()
            fine_scaler.unscale_(fine_optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )
            fine_scaler.step(fine_optimizer)
            fine_scaler.update()

            total_train_loss += loss.item()

            if batch_num % 30 == 0:
                print(
                    f"Fine-tune {epoch}/{EXTRA_EPOCHS} "
                    f"| batch {batch_num}/{len(train_loader)} "
                    f"| loss {total_train_loss/batch_num:.4f}",
                    flush=True
                )

        metrics = evaluate(val_loader, threshold=0.5)
        current_iou = metrics["iou"]

        mlflow.log_metrics({
            "fine_train_loss": total_train_loss / len(train_loader),
            "fine_validation_iou": current_iou,
            "fine_validation_dice": metrics["dice"],
        }, step=epoch)

        print(
            f"\nEpoch {epoch}/{EXTRA_EPOCHS}"
            f" | Validation IoU: {current_iou:.4f}"
            f" | Best: {best_val_iou:.4f}",
            flush=True
        )

        if current_iou > best_val_iou + MIN_IMPROVEMENT:
            best_val_iou = current_iou
            bad_epochs = 0
            best_path = FINETUNED_PATH

            torch.save({
                "state_dict": model.state_dict(),
                "epoch": checkpoint["epoch"] + epoch,
                "best_val_iou": best_val_iou,
                "architecture": "DepthAwareSegFormer",
                "pretrained": PRETRAINED,
                "size": SIZE,
            }, FINETUNED_PATH)

            print("NEW BEST MODEL SAVED!", flush=True)
        else:
            bad_epochs += 1
            print(
                f"No meaningful improvement: "
                f"{bad_epochs}/{PATIENCE}",
                flush=True
            )

        if bad_epochs >= PATIENCE:
            print("EARLY STOPPING triggered")
            break

    mlflow.log_metric("final_best_validation_iou", best_val_iou)

    if best_path == FINETUNED_PATH:
        mlflow.log_artifact(str(FINETUNED_PATH))

print("\nFINE-TUNING COMPLETE!! :)")
print("Best validation IoU:", round(best_val_iou, 4))
print("Best model:", best_path)
print("Time:", round((time.perf_counter()-start)/60, 1), "minutes")

In [0]:

import random
import time
from pathlib import Path

import mlflow
import numpy as np
import torch

SEARCH_DIR = SAVE / "hyperparameter_search"
SEARCH_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATES = [
    SAVE / "glass_segformer_fusion_best.pt",
    SAVE / "glass_segformer_fusion_finetuned.pt",
    SAVE / "glass_segformer_fusion_augmented_v1.pt",
]

source_path = None
baseline_iou = -1.0

for path in CANDIDATES:
    if not path.exists():
        continue

    state = torch.load(
        path,
        map_location="cpu",
        weights_only=True
    )

    model.load_state_dict(
        state["state_dict"],
        strict=True
    )

    score = evaluate(
        val_loader,
        threshold=0.5
    )["iou"]

    print(
        f"Candidate {path.name}: "
        f"validation IoU = {score:.4f}",
        flush=True
    )

    if score > baseline_iou:
        baseline_iou = score
        source_path = path

assert source_path is not None, "No checkpoint found"

source_checkpoint = torch.load(
    source_path,
    map_location="cpu",
    weights_only=True
)

source_state = {
    k: v.detach().cpu().clone()
    for k, v in source_checkpoint["state_dict"].items()
}

trials = [
    {
        "encoder_lr": 3e-6,
        "head_lr": 3e-5,
        "weight_decay": 1e-4
    },
    {
        "encoder_lr": 3e-6,
        "head_lr": 8e-5,
        "weight_decay": 3e-4
    },
    {
        "encoder_lr": 8e-6,
        "head_lr": 3e-5,
        "weight_decay": 3e-4
    },
    {
        "encoder_lr": 8e-6,
        "head_lr": 8e-5,
        "weight_decay": 1e-4
    },
    {
        "encoder_lr": 1.5e-5,
        "head_lr": 5e-5,
        "weight_decay": 1e-4
    },
    {
        "encoder_lr": 1.5e-5,
        "head_lr": 1.2e-4,
        "weight_decay": 3e-4
    },
]

EPOCHS_PER_TRIAL = 4
PATIENCE = 2
MIN_GAIN = 0.001

search_loader = (
    augmented_loader
    if "augmented_loader" in globals()
    else train_loader
)

GLOBAL_BEST_PATH = (
    SEARCH_DIR / "glass_segformer_hpo_best.pt"
)

global_best_iou = baseline_iou
results = []
search_start = time.perf_counter()

print(
    f"Starting from {source_path.name} "
    f"| validation IoU {baseline_iou:.4f}"
)

print(
    "Training loader:",
    "augmented"
    if search_loader is not train_loader
    else "original"
)

for number, settings in enumerate(trials, 1):

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    model.load_state_dict(
        source_state,
        strict=True
    )

    model.to(DEVICE)

    optimizer_hpo = torch.optim.AdamW(
        [
            {
                "params": model.rgb_encoder.parameters(),
                "lr": settings["encoder_lr"]
            },
            {
                "params": [
                    p
                    for name, p in model.named_parameters()
                    if not name.startswith("rgb_encoder.")
                ],
                "lr": settings["head_lr"]
            }
        ],
        weight_decay=settings["weight_decay"]
    )

    scheduler_hpo = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer_hpo,
            T_max=EPOCHS_PER_TRIAL,
            eta_min=1e-7
        )
    )

    scaler_hpo = torch.amp.GradScaler("cuda")
    trial_best = baseline_iou
    stale = 0

    print(
        f"\nTRIAL {number}/{len(trials)} | {settings}",
        flush=True
    )

    with mlflow.start_run(
        run_name=f"glass_hpo_trial_{number:02d}",
        nested=mlflow.active_run() is not None
    ):

        mlflow.log_params({
            **settings,
            "starting_checkpoint": source_path.name,
            "epochs_max": EPOCHS_PER_TRIAL,
            "patience": PATIENCE,
            "augmentation": (
                search_loader is not train_loader
            )
        })

        for epoch in range(1, EPOCHS_PER_TRIAL + 1):

            model.train()
            losses = []

            for x, y in search_loader:

                x = x.to(
                    DEVICE,
                    non_blocking=True
                )

                y = y.to(
                    DEVICE,
                    non_blocking=True
                )

                optimizer_hpo.zero_grad(
                    set_to_none=True
                )

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):
                    logits = model(x)
                    loss = total_loss(logits, y)

                scaler_hpo.scale(loss).backward()
                scaler_hpo.unscale_(optimizer_hpo)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0
                )

                scaler_hpo.step(optimizer_hpo)
                scaler_hpo.update()

                losses.append(float(loss.item()))

            scheduler_hpo.step()

            metrics = evaluate(
                val_loader,
                threshold=0.5
            )

            iou = metrics["iou"]

            mlflow.log_metrics({
                "train_loss": float(np.mean(losses)),
                "validation_iou": iou,
                "validation_dice": metrics["dice"]
            }, step=epoch)

            print(
                f"Trial {number}"
                f" | epoch {epoch}/{EPOCHS_PER_TRIAL}"
                f" | val IoU {iou:.4f}"
                f" | global best {global_best_iou:.4f}",
                flush=True
            )

            if iou > trial_best + MIN_GAIN:
                trial_best = iou
                stale = 0
            else:
                stale += 1

            if iou > global_best_iou + MIN_GAIN:

                global_best_iou = iou

                torch.save({
                    "state_dict": model.state_dict(),
                    "architecture": "DepthAwareSegFormer",
                    "pretrained": PRETRAINED,
                    "size": SIZE,
                    "best_val_iou": global_best_iou,
                    "source_checkpoint": source_path.name,
                    "trial": number,
                    "epoch": epoch,
                    "hyperparameters": settings
                }, GLOBAL_BEST_PATH)

                print(
                    "NEW GLOBAL BEST SAVED",
                    flush=True
                )

            if stale >= PATIENCE:
                print(
                    "Trial early stopped",
                    flush=True
                )
                break

        mlflow.log_metric(
            "trial_best_validation_iou",
            trial_best
        )

        results.append({
            "trial": number,
            **settings,
            "best_val_iou": trial_best
        })

print("\nHYPERPARAMETER SEARCH COMPLETE")

for result in sorted(
    results,
    key=lambda x: x["best_val_iou"],
    reverse=True
):
    print(result)

print(
    f"Starting validation IoU: {baseline_iou:.4f}"
)

print(
    f"Best validation IoU: {global_best_iou:.4f}"
)

selected_path = (
    GLOBAL_BEST_PATH
    if GLOBAL_BEST_PATH.exists()
    else source_path
)

print("Best checkpoint:", selected_path)

print(
    "Elapsed minutes:",
    round(
        (time.perf_counter() - search_start) / 60,
        2
    )
)

selected_state = torch.load(
    selected_path,
    map_location="cpu",
    weights_only=True
)

model.load_state_dict(
    selected_state["state_dict"],
    strict=True
)

model.to(DEVICE)


In [0]:
%pip install coremltools "transformers==4.57.6"

In [0]:
%restart_python

In [0]:
%pip install coremltools "transformers==4.57.6" safetensors

In [0]:

import json
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import SegformerConfig, SegformerModel

CHECKPOINT = Path(
    "/Volumes/workspace/default/glass_sample/models/"
    "rgbd_glass_segformer_fusion/hyperparameter_search/"
    "glass_segformer_hpo_best.pt"
)

EXPORT_DIR = CHECKPOINT.parent

assert CHECKPOINT.exists(), f"Checkpoint not found: {CHECKPOINT}"

class DepthAwareSegFormer(nn.Module):
    def __init__(self):
        super().__init__()

        cfg = SegformerConfig()
        cfg._attn_implementation = "eager"

        self.rgb_encoder = SegformerModel(cfg)

        widths = list(cfg.hidden_sizes)
        projection_width = 64
        fused_width = 96

        self.rgb_projections = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(w, projection_width, 1),
                nn.GroupNorm(8, projection_width),
                nn.GELU()
            )
            for w in widths
        ])

        self.rgb_fuse = nn.Sequential(
            nn.Conv2d(
                4 * projection_width,
                fused_width,
                3,
                padding=1
            ),
            nn.GroupNorm(8, fused_width),
            nn.GELU()
        )

        self.depth_encoder = nn.Sequential(
            nn.Conv2d(2, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, fused_width, 3, padding=1),
            nn.GroupNorm(8, fused_width),
            nn.GELU()
        )

        self.depth_gate = nn.Sequential(
            nn.Conv2d(
                fused_width * 2 + 1,
                fused_width,
                3,
                padding=1
            ),
            nn.Sigmoid()
        )

        self.classifier = nn.Sequential(
            nn.Conv2d(
                fused_width * 2,
                fused_width,
                3,
                padding=1
            ),
            nn.GroupNorm(8, fused_width),
            nn.GELU(),
            nn.Dropout2d(0.1),
            nn.Conv2d(fused_width, 1, 1)
        )

    def forward(self, x):
        rgb = x[:, :3]
        depth = x[:, 3:4]
        valid = x[:, 4:5]

        hidden = self.rgb_encoder(
            pixel_values=rgb,
            output_hidden_states=True,
            return_dict=True
        ).hidden_states

        target = hidden[0].shape[-2:]

        multiscale = [
            F.interpolate(
                proj(h),
                size=target,
                mode="bilinear",
                align_corners=False
            )
            for proj, h in zip(
                self.rgb_projections,
                hidden
            )
        ]

        rgb_features = self.rgb_fuse(
            torch.cat(multiscale, dim=1)
        )

        depth_features = self.depth_encoder(
            torch.cat([depth, valid], dim=1)
        )

        depth_features = F.interpolate(
            depth_features,
            size=target,
            mode="bilinear",
            align_corners=False
        )

        valid_small = F.interpolate(
            valid,
            size=target,
            mode="nearest"
        )

        gate = self.depth_gate(
            torch.cat(
                [
                    rgb_features,
                    depth_features,
                    valid_small
                ],
                dim=1
            )
        )

        fused = torch.cat(
            [
                rgb_features,
                gate * depth_features
            ],
            dim=1
        )

        logits_small = self.classifier(fused)

        return F.interpolate(
            logits_small,
            size=x.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

model = DepthAwareSegFormer().eval()

model.load_state_dict(
    checkpoint["state_dict"],
    strict=True
)

example = torch.zeros(
    1, 5, 224, 224,
    dtype=torch.float32
)

example[:, 3, :, :] = 0.5
example[:, 4, :, :] = 1.0

with torch.inference_mode():
    reference = model(example)

    traced = torch.jit.trace(
        model,
        example,
        strict=False,
        check_trace=False
    )

    traced.eval()
    traced_output = traced(example)

assert tuple(reference.shape) == (1, 1, 224, 224)

max_error = (
    reference - traced_output
).abs().max().item()

assert max_error < 1e-3, (
    f"TorchScript output mismatch: {max_error}"
)

TS_PATH = (
    EXPORT_DIR / "GlassSegmentation_torchscript.pt"
)

traced.save(str(TS_PATH))

print(
    "TORCHSCRIPT OK:",
    TS_PATH,
    "max error:",
    max_error,
    flush=True
)

config = {
    "architecture": (
        "SegFormer-B0 with custom "
        "depth-validity gated fusion"
    ),
    "source_checkpoint": str(CHECKPOINT),
    "input_name": "rgbd",
    "input_shape": [1, 5, 224, 224],
    "input_channels": [
        "R",
        "G",
        "B",
        "depth_normalized",
        "depth_valid"
    ],
    "input_dtype": "float32",
    "rgb_input": (
        "RGB/255 then channelwise "
        "ImageNet normalization"
    ),
    "rgb_mean": [0.485, 0.456, 0.406],
    "rgb_std": [0.229, 0.224, 0.225],
    "depth_input": (
        "Resize nearest, valid=isfinite(D) "
        "and D>0, normalize by p99(valid D) "
        "per frame, clip [0,1], invalid=0"
    ),
    "validity": (
        "1 if finite depth > 0, else 0"
    ),
    "output_name": "logits",
    "output_shape": [1, 1, 224, 224],
    "output_processing": (
        "sigmoid(logits) then apply "
        "validation-selected threshold"
    ),
    "suggested_initial_threshold": 0.5,
    "deployment_warning": (
        "Experimental; iPhone LiDAR domain "
        "shift and glass distance uncertainty "
        "require validation"
    )
}

config_path = (
    EXPORT_DIR / "GlassSegmentation_config.json"
)

config_path.write_text(
    json.dumps(config, indent=2)
)

import coremltools as ct

print(
    "COREMLTOOLS:",
    ct.__version__,
    flush=True
)

mlmodel = ct.convert(
    traced,
    source="pytorch",
    convert_to="mlprogram",
    inputs=[
        ct.TensorType(
            name="rgbd",
            shape=(1, 5, 224, 224),
            dtype=np.float32
        )
    ],
    outputs=[
        ct.TensorType(
            name="logits",
            dtype=np.float32
        )
    ],
    minimum_deployment_target=ct.target.iOS16,
    compute_precision=ct.precision.FLOAT16,
    skip_model_load=True
)

package_dir = Path(
    "/tmp/GlassSegmentation.mlpackage"
)

if package_dir.exists():
    shutil.rmtree(package_dir)

mlmodel.save(str(package_dir))

zip_path = Path(
    shutil.make_archive(
        "/tmp/GlassSegmentation",
        "zip",
        root_dir="/tmp",
        base_dir="GlassSegmentation.mlpackage"
    )
)

volume_zip = (
    EXPORT_DIR / "GlassSegmentation.zip"
)

shutil.copy2(zip_path, volume_zip)

print(
    "CORE ML SUCCESS:",
    volume_zip,
    flush=True
)

print(
    "Download ZIP, unzip on Mac, "
    "drag .mlpackage into Xcode.",
    flush=True
)

print(
    "Config:",
    config_path,
    flush=True
)
